# DLinear


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project


In [ ]:
import os
if not os.path.exists('/content/ml_final_project'):
    !git clone -q https://github.com/ochiga07/ml_final_project.git /content/ml_final_project
import sys
sys.path.append('/content/ml_final_project')

from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline

from metrics import wmae


In [ ]:
!pip install -q mlflow dagshub


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mlflow
import dagshub
import warnings
warnings.filterwarnings('ignore')

if 'DAGSHUB_USER_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')
    except Exception:
        pass

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment('DLinear_Training')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


## Data


In [ ]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)

train_df = full_df[full_df['is_train'] == 1].drop(columns=['is_train'])
print(train_df.shape)


In [ ]:
pairs = train_df.groupby(['Store', 'Dept'])
print(f'total pairs: {len(pairs)}')

LOOKBACK = 12
HORIZON = 1
MIN_LEN = LOOKBACK + HORIZON + 4

series_list = []
pair_keys = []

for (store, dept), grp in pairs:
    grp = grp.sort_values('Date')
    sales = grp['Weekly_Sales'].values
    holidays = grp['IsHoliday'].values
    if len(sales) < MIN_LEN:
        continue
    series_list.append((sales, holidays))
    pair_keys.append((store, dept))

print(f'useable pairs: {len(series_list)}')


In [ ]:
with mlflow.start_run(run_name='DLinear_Preprocessing'):
    mlflow.log_param('lookback', LOOKBACK)
    mlflow.log_param('horizon', HORIZON)
    mlflow.log_param('pairs_kept', len(series_list))
    print('preprocessing logged')


## Dataset


In [ ]:
class SalesDataset(Dataset):
    def __init__(self, series_list, lookback, horizon, train=True, val_weeks=12):
        self.samples = []
        self.holidays = []
        for sales, hols in series_list:
            n = len(sales)
            if train:
                end = n - val_weeks
            else:
                end = n

            start_from = 0 if train else n - val_weeks - lookback
            start_from = max(start_from, 0)

            for i in range(start_from, end - lookback - horizon + 1):
                x = sales[i:i + lookback]
                y = sales[i + lookback:i + lookback + horizon]
                h = hols[i + lookback:i + lookback + horizon]
                if not train and i + lookback < n - val_weeks:
                    continue
                self.samples.append((x, y))
                self.holidays.append(h)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.FloatTensor(x), torch.FloatTensor(y)


In [ ]:
VAL_WEEKS = 12

train_ds = SalesDataset(series_list, LOOKBACK, HORIZON, train=True, val_weeks=VAL_WEEKS)
val_ds = SalesDataset(series_list, LOOKBACK, HORIZON, train=False, val_weeks=VAL_WEEKS)

print(f'train samples: {len(train_ds)}')
print(f'val samples: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False)


## Model


In [ ]:
class MovingAvg(nn.Module):
    # moving avg for trend decomposition
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        padding = (kernel_size - 1) // 2
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=padding)

    def forward(self, x):
        out = self.avg(x.unsqueeze(1)).squeeze(1)
        return out


class DLinear(nn.Module):
    def __init__(self, lookback, horizon, kernel_size=3):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.decomp = MovingAvg(kernel_size)

        self.linear_trend = nn.Linear(lookback, horizon)
        self.linear_seasonal = nn.Linear(lookback, horizon)

    def forward(self, x):
        trend = self.decomp(x)
        seasonal = x - trend
        return self.linear_trend(trend) + self.linear_seasonal(seasonal)


## Training


In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
    return total_loss / n


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    n = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            total_loss += loss.item() * x.size(0)
            n += x.size(0)
    return total_loss / n


def compute_val_wmae(model, val_ds):
    model.eval()
    all_preds = []
    all_true = []
    all_hols = []
    with torch.no_grad():
        for i in range(len(val_ds)):
            x, y = val_ds[i]
            pred = model(x.unsqueeze(0).to(device))
            all_preds.append(pred.cpu().numpy().flatten())
            all_true.append(y.numpy().flatten())
            all_hols.append(val_ds.holidays[i])

    preds = np.concatenate(all_preds)
    true = np.concatenate(all_true)
    hols = np.concatenate(all_hols)
    return wmae(true, preds, hols)


In [ ]:
PARAMS = {
    'lookback': LOOKBACK,
    'horizon': HORIZON,
    'kernel_size': 3,
    'lr': 1e-3,
    'epochs': 30,
    'batch_size': 1024,
}

model = DLinear(
    lookback=PARAMS['lookback'],
    horizon=PARAMS['horizon'],
    kernel_size=PARAMS['kernel_size'],
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=PARAMS['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.L1Loss()

print(f'params: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
with mlflow.start_run(run_name='DLinear_Baseline'):
    mlflow.log_params(PARAMS)

    best_val_loss = float('inf')
    best_state = None

    for epoch in range(PARAMS['epochs']):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss = eval_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        mlflow.log_metric('train_mae', train_loss, step=epoch)
        mlflow.log_metric('val_mae', val_loss, step=epoch)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0:
            print(f'epoch {epoch+1}: train_mae={train_loss:.2f}, val_mae={val_loss:.2f}')

    model.load_state_dict(best_state)
    model.to(device)
    val_wmae_score = compute_val_wmae(model, val_ds)

    mlflow.log_metric('best_val_mae', best_val_loss)
    mlflow.log_metric('val_wmae', val_wmae_score)
    print(f'\nbest val MAE: {best_val_loss:.2f}')
    print(f'val WMAE: {val_wmae_score:.2f}')


## HPO


In [ ]:
configs = [
    {'kernel_size': 3, 'lr': 1e-3},
    {'kernel_size': 5, 'lr': 1e-3},
    {'kernel_size': 7, 'lr': 1e-3},
    {'kernel_size': 3, 'lr': 5e-4},
    {'kernel_size': 5, 'lr': 5e-4},
]

hpo_results = []

with mlflow.start_run(run_name='DLinear_HPO'):
    for i, cfg in enumerate(configs):
        with mlflow.start_run(run_name=f'DLinear_HPO_trial{i}', nested=True):
            mlflow.log_params(cfg)

            m = DLinear(
                lookback=LOOKBACK, horizon=HORIZON,
                kernel_size=cfg['kernel_size'],
            ).to(device)

            opt = torch.optim.Adam(m.parameters(), lr=cfg['lr'])
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
            crit = nn.L1Loss()

            best_vl = float('inf')
            best_st = None
            for epoch in range(25):
                tl = train_epoch(m, train_loader, opt, crit)
                vl = eval_epoch(m, val_loader, crit)
                sched.step(vl)
                if vl < best_vl:
                    best_vl = vl
                    best_st = {k: v.cpu().clone() for k, v in m.state_dict().items()}

            m.load_state_dict(best_st)
            m.to(device)
            score = compute_val_wmae(m, val_ds)

            mlflow.log_metric('best_val_mae', best_vl)
            mlflow.log_metric('val_wmae', score)
            hpo_results.append({**cfg, 'val_wmae': score, 'val_mae': best_vl})
            print(f'trial {i}: wmae={score:.2f}, mae={best_vl:.2f}')

hpo_df = pd.DataFrame(hpo_results).sort_values('val_wmae')
hpo_df


## Final model


In [ ]:
best_cfg = hpo_df.iloc[0]

FINAL_PARAMS = {
    'lookback': LOOKBACK,
    'horizon': HORIZON,
    'kernel_size': int(best_cfg['kernel_size']),
    'lr': float(best_cfg['lr']),
    'epochs': 40,
}

all_ds = SalesDataset(series_list, LOOKBACK, HORIZON, train=True, val_weeks=0)
all_loader = DataLoader(all_ds, batch_size=1024, shuffle=True)

final_model = DLinear(
    lookback=FINAL_PARAMS['lookback'],
    horizon=FINAL_PARAMS['horizon'],
    kernel_size=FINAL_PARAMS['kernel_size'],
).to(device)

opt = torch.optim.Adam(final_model.parameters(), lr=FINAL_PARAMS['lr'])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
crit = nn.L1Loss()

with mlflow.start_run(run_name='DLinear_Final') as run:
    mlflow.log_params(FINAL_PARAMS)

    for epoch in range(FINAL_PARAMS['epochs']):
        tl = train_epoch(final_model, all_loader, opt, crit)
        sched.step(tl)
        mlflow.log_metric('train_mae', tl, step=epoch)
        if (epoch + 1) % 10 == 0:
            print(f'epoch {epoch+1}: train_mae={tl:.2f}')

    mlflow.log_metric('val_wmae', float(best_cfg['val_wmae']))

    torch.save(final_model.state_dict(), f'{drive_repo}/dlinear_final.pt')
    mlflow.log_artifact(f'{drive_repo}/dlinear_final.pt')

    import sys
    if f'{drive_repo}/src' not in sys.path:
        sys.path.append(f'{drive_repo}/src')
    from walmart_pytorch import WalmartPyTorchPipeline
    import mlflow.sklearn

    pipeline = WalmartPyTorchPipeline(model=final_model, lookback=LOOKBACK, device=device)
    pipeline.set_raw_data(train, features, stores)

    mlflow.sklearn.log_model(
        pipeline,
        'pytorch_pipeline',
        code_paths=[f'{drive_repo}/src/walmart_pytorch.py'],
            serialization_format='cloudpickle'
    )
    print(f'model saved, best hpo wmae = {best_cfg["val_wmae"]:.2f}')


## Register model


In [ ]:
run_id = mlflow.search_runs(
    experiment_names=['DLinear_Training'],
    filter_string="tags.mlflow.runName = 'DLinear_Final'",
    order_by=['start_time DESC'],
    max_results=1,
)['run_id'].iloc[0]

model_uri = f'runs:/{run_id}/pytorch_pipeline'
mv = mlflow.register_model(model_uri, 'Walmart_DLinear')
print(f'Model registered: {mv.name} v{mv.version}')


In [ ]:
loaded = mlflow.sklearn.load_model(model_uri)
raw_test = test[['Store', 'Dept', 'Date', 'IsHoliday']].head(100)
preds = loaded.predict(raw_test)
print(f'Raw test predict ok, got {len(preds)} predictions')
